# Exploratory Data Analysis

### Everything & Then Some — *Will They Return, and How Much Will They Spend?*

| | |
|:--|:--|
| **Course** | Exploratory Topics in AI (ETAI) · 2026/27 |
| **Group** | 8 |
| **Author** | Khadija Belhamra |
| **Data** | `train.csv` — 6,534 purchases, Jul 2015 – Feb 2025 · `test.csv` — 986 purchases, May 2025 – Oct 2026 |

---

## Context

Everything & Then Some, a wholesale supplier, wants to know after each purchase whether the customer
will buy again within 90 days (`repeat_purchase_90d`) and, if so, how much they will spend
(`spend_next_90d`). Each row of the data is one purchase, described by the invoice and by the
customer's purchase history at that moment.

## Purpose

This notebook covers **Objective 1 — data exploration and cleaning**, in two stages:

1. **Exploration** — What do the data contain, and how are they structured?
2. **Diagnosis and cleaning** — Which values are missing, invalid, inconsistent or duplicated, and
   how is each error corrected?

Preprocessing (imputation, outliers, feature engineering, encoding, scaling and feature selection)
is studied in a separate notebook, on the cleaned data produced here.

## Method

Every issue goes through the same cycle:

1. **Check** — a function of `src/diagnostics.py` measures the issue; checks only report and never
   change the data.
2. **Finding → decision** — the evidence leads to a decision, written as a rule in `config.yaml`.
3. **Fix** — a function of `src/cleaning.py` applies the rule.
4. **Verification** — the check is run again to confirm the issue is solved and nothing else changed.

The decisions are summarised at the end of the notebook; together, they define the cleaning step of
the pipeline (`clean_dataset`).

## Principles

- **Training data only.** `test.csv` stands for future, unseen purchases. It is used only to check
  that it has the same columns as `train.csv`; every other analysis and decision relies on
  `train.csv` alone, so that no information from the test period influences the pipeline.
- **Row-by-row cleaning.** Cleaning corrects values that are wrong by definition, using fixed rules
  and only the row being corrected. It learns nothing from the data, so it applies identically to
  the training data, to `test.csv` and to a single new purchase.

## Reproducibility

Run from the `notebooks/` folder with the project environment (`.venv`, see the README), all cells
from top to bottom. The notebook reads its settings from `config.yaml` and its functions from
`src/`, so it reflects the current state of the pipeline.

<a id="section-0"></a>

## 0. Setup

Libraries, project functions and raw data. The project's own functions are imported from `src/`,
and all paths and rules are read from `config.yaml`, so the notebook and the pipeline always use
the same code and settings.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

# The notebook runs from notebooks/: the project root is one folder up
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

pd.set_option("display.max_columns", None)

In [2]:
import numpy as np

from src.config import load_yaml_config
from src.data import load_data
from src import diagnostics as diag

from src.cleaning import *

Both files are loaded as they are, without any correction: every change to the data is decided
and justified in the sections below.

In [3]:
config = load_yaml_config(project_root / "config.yaml")
cleaning_config = config["cleaning"]["train"]   # the rules used for train.csv and test.csv

train_raw = load_data(project_root / config["data"]["csv_paths"]["train"])
test_raw = load_data(project_root / config["data"]["csv_paths"]["test"])
sample_submission = load_data(project_root / config["data"]["csv_paths"]["sample_submission"])

n_total = len(train_raw) + len(test_raw)

print("train:", train_raw.shape, "| test:", test_raw.shape)
print(f"train: {len(train_raw) / n_total:.1%} | test: {len(test_raw) / n_total:.1%} of {n_total} rows")

train: (6534, 40) | test: (986, 38)
train: 86.9% | test: 13.1% of 7520 rows


<a id="section-1"></a>

## 1. Data cleaning

The cleaning corrects values that are wrong by definition, using fixed rules. Each step follows the
cycle described in the header: check → finding → decision → fix → verification.

<a id="section-1-1"></a>

### 1.1 Schema

Both files must share the same feature columns, so that the pipeline trained on `train.csv` can run
on `test.csv`. The only expected difference is the two targets, which `test.csv` does not contain.

In [4]:
shared_in_train_order = [col for col in train_raw.columns if col in test_raw.columns]

print("Only in train:", sorted(set(train_raw.columns) - set(test_raw.columns)))
print("Only in test:", sorted(set(test_raw.columns) - set(train_raw.columns)))
print("Same order:", shared_in_train_order == list(test_raw.columns))

Only in train: ['repeat_purchase_90d', 'spend_next_90d']
Only in test: []
Same order: True


>As expected, the only columns missing from **test.csv** are the two targets, and the shared columns
come in the same order. <br>From here on, only **train.csv** is used.

In [5]:
# Check the Kaggle submission format
print("Columns:", sample_submission.columns.tolist())
print("Rows:", len(sample_submission), "| test rows:", len(test_raw))
print("Same IDs as test.csv:", set(sample_submission["ID"]) == set(test_raw["ID"]))
sample_submission.head()

Columns: ['ID', 'repeat_purchase_90d']
Rows: 986 | test rows: 986
Same IDs as test.csv: True


,ID,repeat_purchase_90d
0,10000006,1
1,10000008,1
2,10000010,1
3,10000020,0
4,10000025,1


<a id="section-1-2"></a>

### 1.2 First look at the data

In [6]:
# What does a purchase look like?
train_raw.head()

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,repeat_purchase_90d,spend_next_90d,days_since_previous_purchase,customer_tenure_days,prior_purchase_count,prior_total_spend,prior_mean_purchase_value,prior_std_purchase_value,prior_min_purchase_value,prior_max_purchase_value,prior_mean_gap_days,prior_std_gap_days,purchase_count_30d,spend_30d,purchase_count_90d,spend_90d,purchase_count_180d,spend_180d,purchase_count_365d,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,ID,random_noise
0,558.11,510.90,453.75,104.36,0.0,0.0,57.15,153.099911,4.0,1.0,1,359.04,79.0,126.0,2.0,858.30,429.150000,112.656252,349.49,508.81,47.000000,NaN,0.0,0.00,1.0,349.49,2.0,858.30,2.0,858.30,660784,Maybe Tomorrow Payment 10,VLOOKUP Master 9,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2024-10-29,10000000,-0.660448
1,978.80,830.02,795.77,183.03,0.0,0.0,34.25,472.943134,32.0,1.0,1,592.71,244.0,616.0,5.0,6722.83,1344.566000,1683.795178,84.38,4222.15,93.000000,117.657696,0.0,0.00,0.0,0.00,0.0,0.00,4.0,2500.68,189146,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 36,Final Excel Warehouse 1,Ctrl Z Series 1,2017-07-08,10000001,0.924510
2,10.33,12.00,8.40,1.93,0.0,0.0,3.60,6.097485,1.0,1.0,1,423.72,9.0,610.0,90.0,17590.08,195.445333,190.389932,4.54,1002.15,6.752809,6.384073,2.0,111.27,9.0,1928.55,28.0,5290.25,49.0,7899.80,103940,Maybe Tomorrow Payment 9,VLOOKUP Master 1,Decorative Dashboard Zone 4,NaN,Final Excel Warehouse 1,Ctrl Z Series 3,2019-03-19,10000002,-0.258730
3,856.83,696.61,696.61,160.22,0.0,0.0,0.00,408.167275,9.0,1.0,0,NaN,97.0,97.0,1.0,270.33,270.330000,NaN,270.33,270.33,NaN,NaN,0.0,0.00,0.0,0.00,1.0,270.33,1.0,270.33,512927,Maybe Tomorrow Payment 7,VLOOKUP Master 4,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 71,Final Excel Warehouse 1,Ctrl Z Series 3,2018-11-10,10000003,1.448402
4,1616.10,1488.85,1313.90,302.20,0.0,0.0,174.95,485.401864,33.0,1.0,1,455.73,85.0,1575.0,25.0,12803.02,512.120800,406.696963,22.58,1736.74,62.083333,66.246520,0.0,0.00,2.0,184.13,4.0,2062.50,4.0,2062.50,122356,Maybe Tomorrow Payment 12,NaN,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2021-11-30,10000004,-1.058890


In [7]:
# Check column names
print("Column names:")
print(train_raw.columns.tolist())

Column names:
['Total a Pagar', 'Total Bruto', 'Total Liquido', 'Total Imp_', 'Total Portes', 'Total Desc_ Global', 'Total Desc_ Linha', 'Custo', 'N_ Detalhes', 'invoice_count_current_purchase', 'repeat_purchase_90d', 'spend_next_90d', 'days_since_previous_purchase', 'customer_tenure_days', 'prior_purchase_count', 'prior_total_spend', 'prior_mean_purchase_value', 'prior_std_purchase_value', 'prior_min_purchase_value', 'prior_max_purchase_value', 'prior_mean_gap_days', 'prior_std_gap_days', 'purchase_count_30d', 'spend_30d', 'purchase_count_90d', 'spend_90d', 'purchase_count_180d', 'spend_180d', 'purchase_count_365d', 'spend_365d', 'customer_id', 'payment_type', 'salesperson', 'commercial_zone', 'transport_method', 'warehouse', 'document_series', 'purchase_date', 'ID', 'random_noise']


<a id="section-1-2"></a>

### 1.2 Column names

Nine columns have Portuguese names with spaces, capitals and abbreviations (`Total a Pagar`, `Total Imp_`, `N_ Detalhes`, ...), while all the others are English `snake_case`.

Therefore, we decide to rename these nine columns to English `snake_case` so every column follows the same convention. `ID` keeps its name since it is the column name the Kaggle submission file expects.
The mapping is stored in `config.yaml` under `cleaning → rename_columns`, so the pipeline renames the columns the same way. We work on a copy `df` from here on and leave `train_raw` untouched.

In [8]:
# Make a copy of train_raw for further cleaning and renaming so we don't modify the original DataFrame
df = train_raw.copy()
apply_column_renames(df, cleaning_config["rename_columns"])

pd.Series(cleaning_config["rename_columns"], name="new name").rename_axis("original name").to_frame()

,new name
original name,
Total a Pagar,total_amount_payable
Total Bruto,gross_total
Total Liquido,net_total
Total Imp_,total_tax
Total Portes,total_shipping
Total Desc_ Global,total_global_discount
Total Desc_ Linha,total_line_item_discount
Custo,cost
N_ Detalhes,details_count


In [9]:
# Columns that are still not in snake_case (only ID is expected)
[col for col in df.columns if col != col.lower().replace(" ", "_")]

['ID']

<a id="section-1-3"></a>

### 1.3 Types ###

In [10]:
# DataFrame info: types, non-null counts, memory usage
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6534 entries, 0 to 6533
Data columns (total 40 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   total_amount_payable            6469 non-null   float64
 1   gross_total                     6469 non-null   float64
 2   net_total                       6469 non-null   float64
 3   total_tax                       6469 non-null   float64
 4   total_shipping                  6469 non-null   float64
 5   total_global_discount           6469 non-null   float64
 6   total_line_item_discount        6469 non-null   float64
 7   cost                            6469 non-null   float64
 8   details_count                   6469 non-null   float64
 9   invoice_count_current_purchase  6469 non-null   float64
 10  repeat_purchase_90d             6534 non-null   int64  
 11  spend_next_90d                  3425 non-null   float64
 12  days_since_previous_purchase    5673 non-null

In [11]:
# How are the values spread?
df.describe().T

,count,mean,std,min,25%,50%,75%,max
total_amount_payable,6469.0,9.255563e+02,1.334998e+03,9.300000e-01,1.889200e+02,4.767400e+02,1.087670e+03,2.229721e+04
gross_total,6469.0,8.861527e+02,1.327641e+03,8.300000e-01,1.757400e+02,4.457400e+02,1.036700e+03,2.157828e+04
net_total,6469.0,7.733364e+02,1.136356e+03,7.600000e-01,1.543800e+02,3.899100e+02,9.033700e+02,1.812781e+04
total_tax,6469.0,1.522199e+02,2.252624e+02,0.000000e+00,2.993000e+01,7.974000e+01,1.848400e+02,4.169400e+03
total_shipping,6469.0,1.151427e+00,5.844091e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.810600e+02
total_global_discount,6469.0,3.499072e-01,5.116564e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.184900e+02
total_line_item_discount,6469.0,1.124664e+02,2.899863e+02,0.000000e+00,0.000000e+00,1.313000e+01,9.228000e+01,4.103440e+03
cost,6469.0,3.281650e+02,5.195072e+02,0.000000e+00,4.905000e+01,1.525400e+02,3.960615e+02,7.427401e+03
details_count,6469.0,2.152450e+01,2.771342e+01,1.000000e+00,4.000000e+00,1.100000e+01,2.900000e+01,4.230000e+02
invoice_count_current_purchase,6469.0,1.066007e+00,3.108069e-01,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,5.000000e+00


- Most columns have 6,469 values out of 6,534, so the same 65 rows seem to be almost empty. 
- `purchase_date` is stored as text.
- Several counts are stored as floats. 

To find which numeric columns are really integers, we look for those whose values are all whole numbers.

In [12]:
# Numeric columns where every non-missing value is a whole number
numeric = df.select_dtypes("number")
is_whole = ((numeric % 1 == 0) | numeric.isna()).all()

print("Numeric columns with whole numbers only: ",is_whole.sum())
pd.DataFrame({
    "type": numeric.dtypes[is_whole],
    "missing": numeric.isna().sum()[is_whole],
})

Numeric columns with whole numbers only:  12


,type,missing
details_count,float64,65
invoice_count_current_purchase,float64,65
repeat_purchase_90d,int64,0
days_since_previous_purchase,float64,861
customer_tenure_days,float64,65
prior_purchase_count,float64,65
purchase_count_30d,float64,65
purchase_count_90d,float64,65
purchase_count_180d,float64,65
purchase_count_365d,float64,65


The checks above showed that two types of conversion were needed:

- **Integers:** There are twelve columns containing only whole numbers, and all of them are integers by nature: counts, durations in whole days, the binary target and the identifiers. Nine are stored as decimals only because they contain missing values, which a plain integer column cannot hold. We convert them using the pandas' nullable integer type (`Int64`) so that existing missing values are preserved.
- **Dates:** For `purchase_date`, we first remove any surrounding spaces and then convert the values using the format specified in `cleaning → date_format` (`%Y-%m-%d`). If a date cannot be parsed, it is set to missing (`NaT`) rather than causing an error.

These conversions are applied by `apply_correct_types` in `src/cleaning.py` which updates the DataFrame in place and leaves all other columns unchanged. Text columns are kept as they are, since no type conversion is needed. The table below compares each converted column before and after cleaning, including its data type and number of missing values, which should remain unchanged.

In [13]:
column_types = cleaning_config["column_types"]
converted_columns = column_types["int"] + column_types["date"]

types_before = df[converted_columns].dtypes.astype(str)
missing_before = df[converted_columns].isna().sum()

apply_correct_types(df, column_types, cleaning_config["date_format"])

types_after = df[converted_columns].dtypes.astype(str)
changed = types_before.str.lower() != types_after.str.lower()   # int64 -> Int64 is not a real change

pd.DataFrame({
    "type before": types_before,
    "type after": types_after,
    "missing before": missing_before,
    "missing after": df[converted_columns].isna().sum(),
})[changed]

,type before,type after,missing before,missing after
details_count,float64,Int64,65,65
invoice_count_current_purchase,float64,Int64,65,65
days_since_previous_purchase,float64,Int64,861,861
customer_tenure_days,float64,Int64,65,65
prior_purchase_count,float64,Int64,65,65
purchase_count_30d,float64,Int64,65,65
purchase_count_90d,float64,Int64,65,65
purchase_count_180d,float64,Int64,65,65
purchase_count_365d,float64,Int64,65,65
purchase_date,str,datetime64[us],65,65


<a id="section-1-4"></a>

### 1.4 Invalid values

A value is invalid when it is impossible on its own, whatever the other columns say: a negative amount, an invoice with no lines, a code used instead of a real value. The minimum and maximum of each numeric column are a first place to look.

In [14]:
# How are the values spread? The min and max reveal impossible values
df.describe().T

,count,mean,min,25%,50%,75%,max,std
total_amount_payable,6469.0,925.556262,0.93,188.92,476.74,1087.67,22297.21,1334.997733
gross_total,6469.0,886.152705,0.83,175.74,445.74,1036.7,21578.28,1327.641086
net_total,6469.0,773.336384,0.76,154.38,389.91,903.37,18127.81,1136.355723
total_tax,6469.0,152.219878,0.0,29.93,79.74,184.84,4169.4,225.262375
total_shipping,6469.0,1.151427,0.0,0.0,0.0,0.0,181.06,5.844091
total_global_discount,6469.0,0.349907,0.0,0.0,0.0,0.0,218.49,5.116564
total_line_item_discount,6469.0,112.466414,0.0,0.0,13.13,92.28,4103.44,289.986307
cost,6469.0,328.16504,0.0,49.05,152.54,396.061499,7427.400583,519.507218
details_count,6469.0,21.524501,1.0,4.0,11.0,29.0,423.0,27.713419
invoice_count_current_purchase,6469.0,1.066007,1.0,1.0,1.0,1.0,5.0,0.310807


Two things stand out: 

- `spend_365d` and `prior_mean_purchase_value` reach exactly 1,000,000,000, far above their 75th percentile, which looks like a code rather than a real amount. 
- Those same two columns, together with `days_since_previous_purchase` and `prior_mean_gap_days`, also have negative minimums, which is impossible for amounts and durations (`random_noise` is negative too, but it carries no real meaning).

Before treating 1,000,000,000 as an error, we check how often it appears and how it compares with the other values of the same columns.

In [15]:
placeholder_columns = ["prior_mean_purchase_value", "spend_365d"]

summary = {}
for col in placeholder_columns:
    is_1e9 = df[col] == 1e9               # rows where the value is exactly 1,000,000,000
    other_values = df.loc[~is_1e9, col]   # all the other values of the column
    most_common = other_values.value_counts()

    summary[col] = {
        # every distinct value above one million (if 1e9 is the only one, it stands alone)
        "values above 1M": sorted(df.loc[df[col] > 1e6, col].unique()),
        # how many times 1e9 appears
        "rows with 1e9": is_1e9.sum(),
        # the most frequent value apart from 1e9, and how many rows have it
        # (real amounts rarely repeat, so a value repeated many times needs an explanation)
        "most common other value": most_common.index[0],
        "rows with that value": most_common.iloc[0],
        # the largest value apart from 1e9
        "largest other value": other_values.max(),
    }

pd.DataFrame(summary).T

,values above 1M,rows with 1e9,most common other value,rows with that value,largest other value
prior_mean_purchase_value,[1000000000.0],16,24.29,2,72937.163128
spend_365d,[1000000000.0],15,0.0,810,522933.206431


Each numeric column has a valid range, stored in `config.yaml` under `cleaning → domain_rules`: amounts, counts and durations cannot be negative, an invoice has at least one line, and the target is 0 or 1. Number of values breaking each rule:

1,000,000,000 is the only value above one million in both columns. It appears 16 times in `prior_mean_purchase_value` and 15 times in `spend_365d`, while real amounts almost never repeat: apart from 1e9, the most common mean appears only twice. The value 0 is repeated 810 times in `spend_365d`, but that is expected, since many customers made no purchase in the last year. Finally, 1e9 is thousands of times larger than the largest other value (72,937 and 522,933).

>Therefore, we consider that 1,000,000,000 is not a real amount but a placeholder for an unknown value, and we treat it as missing.

The placeholder is stored in `config.yaml` under `cleaning → placeholder_values`, and replaced by a missing value with `apply_placeholder_values` from `src/cleaning.py`. It runs before the range rules, so that no later check sees these codes as real amounts.

In [16]:
placeholder_columns = list(cleaning_config["placeholder_values"])
missing_before = df[placeholder_columns].isna().sum()

replaced = apply_placeholder_values(df, cleaning_config["placeholder_values"])
display(replaced)

pd.DataFrame({
    "missing before": missing_before,
    "missing after": df[placeholder_columns].isna().sum(),
    "values above 1M left": (df[placeholder_columns] > 1e6).sum(),
})

,column,placeholder,replaced
0,prior_mean_purchase_value,[1000000000],16
1,spend_365d,[1000000000],15


,missing before,missing after,values above 1M left
prior_mean_purchase_value,862,878,0
spend_365d,260,275,0


#### Negative values

In [17]:
# Check for negative values in numeric columns, excluding the "random_noise" column
diag.find_negative_values(df, exclude=["random_noise"])

,negative values,distinct negative values,min,max
days_since_previous_purchase,16.0,1.0,-4.000000,-4.000000
prior_mean_purchase_value,16.0,16.0,-2478.658478,-110.147097
prior_mean_gap_days,16.0,1.0,-4.000000,-4.000000
spend_365d,15.0,15.0,-159115.175535,-167.180000


Four columns have negative values but amounts, counts and durations cannot be negative. <br>A negative value is either a wrong number, or the right number with a wrong sign. <br>For two of them we can tell: 
- the spend over 365 days includes the spend over the last 180 days, so the true `spend_365d` must be at least `spend_180d`; 
- the true `prior_mean_purchase_value` is known exactly as `prior_total_spend / prior_purchase_count`.

In [18]:
sign_rules = {
    "spend_365d": {"min_column": "spend_180d"},   # 365 days include the last 180
    "prior_mean_purchase_value": {"numerator": "prior_total_spend", "denominator": "prior_purchase_count"},   # mean = total / count
}
diag.check_sign_errors(df, sign_rules)

,column,negative values,test,match without the sign
0,spend_365d,15,|value| >= spend_180d,15
1,prior_mean_purchase_value,16,|value| == prior_total_spend / prior_purchase_...,16


In both columns every negative value is the right number with a wrong sign, so we keep the number and remove the sign instead of discarding it. The columns are listed in `config.yaml` under `cleaning → sign_flip_columns`, and fixed with `apply_sign_fixes` from `src/cleaning.py`.

In [ ]:
fixed = apply_sign_fixes(df, cleaning_config["sign_flip_columns"])
display(fixed)
print("Negative values left:", (df[cleaning_config["sign_flip_columns"]] < 0).sum().to_dict())

,column,fixed
0,spend_365d,15
1,prior_mean_purchase_value,16


Negative values left: {'spend_365d': 0, 'prior_mean_purchase_value': 0}


In [20]:
# Rows that still contain a negative value after the sign fix
numeric_columns = df.select_dtypes("number").columns.drop("random_noise")   # random_noise has no real meaning
has_negative = (df[numeric_columns] < 0).any(axis=1)

df.loc[has_negative, ["days_since_previous_purchase", "prior_mean_gap_days"]]

,days_since_previous_purchase,prior_mean_gap_days
313,-4,40.000000
354,12,-4.000000
496,-4,NaN
511,-4,290.181818
556,86,-4.000000
627,-4,79.333333
686,-4,108.500000
823,8,-4.000000
1305,10,-4.000000
1498,-4,NaN


The two duration columns, `days_since_previous_purchase` and `prior_mean_gap_days`, each have 16 negative values, all exactly −4, and never in the same row. A single repeated value rather than varied measurements suggests a code for a missing value. <br>To know whether −4 could be a sign error, we use the purchase windows:
- if the previous purchase happened 4 days ago, it falls within the last 30 days, so `purchase_count_30d` must be at least 1;
- we first check that this rule holds in the rows with a valid duration, then apply it to the rows with −4.

In [ ]:
# Check the recency rule: if a customer made a purchase in the last 30 days, they should have at least one purchase in the last 30 days
diag.check_recency_rule(df, "days_since_previous_purchase", "purchase_count_30d", window_days=30)

,rows,purchase_count_30d = 0 (contradiction)
"valid rows, days_since_previous_purchase < 30",2042,0
"negative rows, |days_since_previous_purchase| < 30",16,11


The rule holds in every valid row, but 11 of the 16 rows with −4 days have no purchase in the last 30 days, so the true value cannot be 4. <br>−4 is therefore not a sign error but a code, and the true duration cannot be recovered from the other columns. `prior_mean_gap_days` cannot be checked the same way, since it is an average, but it contains the same code −4, so we treat it the same way. <br>We apply the range rules (`cleaning → domain_rules` in `config.yaml`), which set these values to missing.

In [26]:
violations = apply_domain_rules(df, cleaning_config["domain_rules"])
display(violations[violations["violations"] > 0])

# Check: run the rules again on a copy, nothing should be left
print("Violations left:", apply_domain_rules(df.copy(), cleaning_config["domain_rules"])["violations"].sum())

,column,rule,violations


Violations left: 0


The range rules set the 32 values of −4 to missing, and no value outside its valid range is left.

In total, the invalid values of this section were handled in three ways:
- the 31 placeholders (1,000,000,000) in `prior_mean_purchase_value` (16) and `spend_365d` (15) were set to missing;
- the 31 negative amounts in the same two columns were kept, with their sign corrected;
- the 32 codes of −4 in `days_since_previous_purchase` and `prior_mean_gap_days` (16 each) were set to missing.

In this data only these four columns contained invalid values, but the range rules cover every numeric column (amounts, counts and durations cannot be negative, an invoice has at least one line, the target is 0 or 1). All these rules are stored in `config.yaml` and applied by `clean_dataset`, so any impossible value in new data (`test.csv` or a new purchase) is handled the same way, whatever the column. <br>The values set to missing will be imputed during preprocessing.